# Problem Set: Untangling a museum catalogue's single partOf with Claude
### Chapter 6 — Top-down Ontology Development

**Topic coverage:** Keet §6.1 (foundational ontologies, aligning a domain class by decision questions), §6.2 (the part-whole taxonomy, parthood versus its impostors, transitivity and chaining); course notebooks 01–03 of this chapter. Tools: DSPy + GEPA, LangChain agents, the Chapter 6 toolkit (`classify_partwhole`, `can_chain`) and a diagnostic-questioning MDP.

**Level:** Graduate (practice-first). Combines conceptual analysis with building, measuring and optimising a real Claude-backed agent.

**Total points:** 100 (Part A 18 / Part B 30 / Part C 26 / Part D 26); 53 pts are auto-graded by the check cells, the rest are marked by hand against the rubric. **Estimated effort:** 10–12 hours.

**API budget:** ≈ \$5–12 estimated on `claude-opus-5` for a full run (≈ 180 DSPy calls at ≈ \$0.02–0.05 each, GEPA being the largest line, plus 8 agent runs at ≈ \$0.10–0.25 each). Nothing here was measured on a live run. Re-running unchanged cells is free — DSPy caches identical requests (except in C3, which disables the cache on purpose). Every model call in this notebook is a billed request to Anthropic; the cost lines the problems ask for are part of the answer.

## The brief

The **Aldermoor Museum of Art and Industry** is migrating its collections catalogue
(≈ 41,000 records) into a knowledge graph. The legacy system had exactly one
relationship field, `partOf`, and cataloguers used it for everything: the pendulum
of a clock, a coin in a hoard, the bronze of a cast, a painting on loan, letters in
an archive box, a totem pole in the sculpture garden. The public collections search
treats `partOf` as transitive, and it shows: *"What is part of the East Wing?"*
lists the bronze of *The Thinker*, and the insurance export counts a pendulum as an
item of the clock collection.

The Head of Collections Information has asked you, the ontology engineer, to type
every legacy link with the right part-whole relation. The data office has adopted
the Chapter 6 taxonomy (`ch06_toolkit.PART_WHOLE_RELATIONS`) as the museum's
**modelling convention**: which relations count as parthood and which are
transitive is decided by that table, and that is what "correct" means below. Your
brief:

1. **Measure the damage** a single transitive `partOf` does, and find where the
   chapter's own decision procedure runs out.
2. **Build the grader first**, then a **Claude classifier** that reads a legacy
   entry and returns the categories of both arguments, the relation, and whether it
   is genuine parthood. Measure it on a development split; improve it with GEPA;
   report on a **held-out test split**, with cost and run-to-run noise, next to
   hand-written guidelines and a design that lets the taxonomy decide.
3. **Ship a chaining agent** that answers the search team's *"is X part of Y?"*
   questions with checked chains, never with the old transitive shortcut.
4. **Plan the curator interview** for new classes: model category alignment as a
   stochastic diagnostic-questioning MDP, derive the decision tree from a cost
   model, and find out what it takes for DOLCE's tree to come out on top.

The labelled sample (24 legacy entries, split 8/8/8 by item, every split holding
each of the eight relations once), the menus, the typed catalogue links, the search
questions and the MDP are provided in `ch06_agentic.py`.

## Submission requirements

1. **This notebook, executed top to bottom**, with every `TODO` implemented and every *Your answer* cell written.
2. **The artefacts it saves** under `course/artifacts/` (optimised instructions, run logs) — they are evidence, not by-products.
3. **Honest reporting.** A number without its split, its sample size and its cost earns no credit. Report the runs you did, including the ones that did not improve anything.

The check cells are the marking scheme for the auto-graded points; do not edit them. The final self-test prints your auto-graded subtotal.

> Solutions: `04_solutions.ipynb` — do not open it until you have attempted every problem.

In [ ]:
import sys, json, logging
from pathlib import Path

# Make the course root and this chapter importable no matter where Jupyter started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break
sys.path.insert(0, str(here))

import oe_course
from oe_course import config
from oe_course.grading import Grader

print(json.dumps(oe_course.describe_environment(), indent=1))
config.require_credentials()          # this problem set calls the live Anthropic API
logging.getLogger("dspy").setLevel(logging.WARNING)
GRADER = Grader("Chapter 6 — Top-down Ontology Development — Untangling a museum catalogue's single partOf with Claude")

import re, time
import dspy
import numpy as np
import pandas as pd
import ch06_toolkit as ch6
import ch06_agentic as A
from oe_course import evaluation as ev, llm, mdp, optimize as opt, agents

train, dev, test = A.build_dataset("train"), A.build_dataset("dev"), A.build_dataset("test")
print(f"train {len(train)} / dev {len(dev)} / test {len(test)}")
pd.DataFrame([{"id": ex.id, "split": ex.split, "statement": ex.statement,
               "part": ex.part_category, "whole": ex.whole_category,
               "relation": ex.gold_relation, "parthood": ex.gold_parthood}
              for ex in train + dev + test])

In [ ]:
print("The museum's convention (Chapter 6 toolkit):")
pd.DataFrame([{"relation": r.id, "parthood": r.parthood, "transitive": r.transitive,
               "part": r.part_category, "whole": r.whole_category}
              for r in ch6.PART_WHOLE_RELATIONS])

---

## Part A — What one transitive partOf costs, and where the procedure runs out

The legacy links, now typed, are in `A.CATALOGUE_LINKS` as `(part, relation, whole)`
triples, in the legacy direction (the child was catalogued "part of" the parent). The
live search computes the transitive closure of `partOf` over all of them.

In [ ]:
pd.DataFrame(A.CATALOGUE_LINKS, columns=["part", "relation", "whole"])

### A1. Every spurious answer the search gives (10 pts)

1. Implement `naive_closure(links) -> set[(part, whole)]`: the transitive closure the
   search computes, ignoring the relation column.
2. Implement `sound_closure(links) -> set[(part, relation, whole)]`: the links plus
   every triple derivable by composing `a R1 b` and `b R2 c` into `a R1 c` **only
   when `ch6.can_chain(R1, R2)["valid"]`** (a valid chain has `R1 == R2`), repeated to
   a fixpoint.
3. Build `a1`, one row per pair the search returns that the sound closure does not,
   with columns `part`, `whole`, `path` (the relations along the path, e.g. from
   `A.CatalogueWorkspace().upward(part)`), and `reason` (the `can_chain` reason for the
   first pair on the path that fails).

Then answer in writing: (a) which of these errors would you expect to do the most
harm, and to whom? (b) `located-in` is transitive in the table, yet `can_chain`
refuses `located-in` + `located-in` — is "the Thinker cast is in the East Wing"
therefore false? (c) A colleague proposes to fix the OWL model by declaring
`componentOf`, `memberOf`, … as sub-properties of one transitive `partOf`. Would the
search stop listing the pendulum under the clock collection?

In [ ]:
def naive_closure(links):
    # TODO
    raise NotImplementedError

def sound_closure(links):
    # TODO
    raise NotImplementedError

naive = sound = a1 = None      # TODO

In [ ]:
# Marking scheme for A1 -- do not edit.
with GRADER.check('A1', points=6):
    toy = [("a", "involved-in", "b"), ("b", "involved-in", "c"), ("c", "involved-in", "d"),
           ("x", "component-of", "y"), ("y", "component-of", "z")]
    assert naive_closure(toy) == {("a", "b"), ("b", "c"), ("c", "d"), ("a", "c"), ("b", "d"),
                                  ("a", "d"), ("x", "y"), ("y", "z"), ("x", "z")}
    assert sound_closure(toy) == set(toy) | {("a", "involved-in", "c"), ("b", "involved-in", "d"),
                                             ("a", "involved-in", "d")}
    assert len(naive) == 26 and len(sound) == 19
    assert set(zip(a1["part"], a1["whole"])) == {
        ("bronze", "sculpture-hall"), ("bronze", "east-wing"), ("thinker-cast", "east-wing"),
        ("pendulum", "clock-collection"), ("teacup", "display-case-7"),
        ("teacup", "east-wing"), ("meissen-tea-service", "east-wing")}
    assert a1["reason"].map(lambda r: isinstance(r, str) and len(r) > 0).all()
    # the sound closure must agree with the search team's gold answers
    for qid, part, whole, _, gold in A.CHAIN_QUERIES:
        got = any(a == part and c == whole and ch6.relation_by_id(r).parthood for a, r, c in sound)
        assert ("yes" if got else "no") == gold, qid

**Your answer:**

*(Write your answer here.)*

### A2. Where the categories stop deciding (8 pts)

Chapter 6 claims the relation "follows almost mechanically" from the categories of part
and whole. Test the claim on the procedure itself. For every ordered pair of categories
in `A.ARGUMENT_CATEGORIES` and both values of `separable`, record what
`ch6.classify_partwhole` returns. Build `a2` (50 rows) with columns `part`, `whole`,
`separable`, `relation`, `parthood`, and `fallback` — `True` when the procedure returns
`component-of` although the pair is **not** (`physical-object`, `physical-object`),
i.e. it fell through to its default. Also set `separable_matters` to the set of
`(part, whole)` pairs whose answer changes with `separable`.

In writing: pick **three** fallback or otherwise suspicious rows (look at every row with
`whole == "collection"` too). For each, give a realistic museum catalogue sentence that
lands there, say what the right modelling is, and say what the migration should do with
such entries (the classifier will meet them in the real 41,000 records).

In [ ]:
a2 = None                  # TODO: 50 rows
separable_matters = None   # TODO: a set of (part, whole) pairs

raise NotImplementedError

In [ ]:
# Marking scheme for A2 -- do not edit.
with GRADER.check('A2', points=4):
    assert isinstance(a2, pd.DataFrame) and len(a2) == 50
    assert {"part", "whole", "separable", "relation", "parthood", "fallback"} <= set(a2.columns)
    for _, row in a2.iterrows():
        assert row.relation == ch6.classify_partwhole(row.part, row.whole, row.separable)
    assert separable_matters == {("physical-object", "physical-object")}
    assert set(zip(a2[a2.fallback].part, a2[a2.fallback].whole)) == {
        ("physical-object", "amount-of-matter"), ("collection", "physical-object"),
        ("collection", "amount-of-matter"), ("collection", "process"),
        ("process", "physical-object"), ("process", "amount-of-matter"),
        ("region", "physical-object"), ("region", "amount-of-matter"), ("region", "process")}

**Your answer:**

*(Write your answer here.)*

---

## Part B — Build the grader and the classifier

The classifier reads one legacy entry and returns four fields: `part_category` and
`whole_category` (ids from `A.category_menu()`), `relation` (an id from
`A.relation_menu()`) and `is_parthood` (`true`/`false`). The guidelines a scorer can
report are in `A.PARTWHOLE_RULEBOOK`; `A.RULE_FOR_RELATION` names, for each *correct*
relation, the guideline a wrong answer violated.

In [ ]:
for rule in A.PARTWHOLE_RULEBOOK:
    print(f"{rule.id:30s} {rule.description[:88]}")
print()
print(A.category_menu()); print(); print(A.relation_menu())

### B1. A diagnostic scorer that separates naming from parthood (14 pts)

Implement `norm_id(text) -> str` (lower-case, strip surrounding whitespace, quotes,
backticks and a trailing full stop, and turn runs of spaces/underscores into single
hyphens, so `"Amount of Matter"` → `"amount-of-matter"`) and
`partwhole_scorer(gold, pred) -> ev.ScoreReport`:

| stage | outcome | score | `violated` |
|---|---|---|---|
| format | a category not in `A.ARGUMENT_CATEGORIES`, a relation not in `A.RELATION_IDS`, or `is_parthood` not one of true/false/yes/no (after `norm_id`) | 0.0 | `["use-the-menus"]` |
| content | otherwise: `0.1·part_ok + 0.1·whole_ok + 0.4·relation_ok + 0.4·parthood_ok` | — | in this order: `"identify-categories-first"` if either category is wrong; `A.RULE_FOR_RELATION[gold.gold_relation]` if the relation is wrong; `"check-genuine-parthood"` if the parthood verdict is wrong |

The notes are what GEPA reads, so make them useful: when the relation is wrong, include
the correct relation's **test question** (`ch6.relation_by_id(...).test`); when the
answer contradicts itself — the relation it named has a parthood flag different from its
own `is_parthood` — say so with the word **"contradicts"**.

Why weight parthood as heavily as the relation name? Because only the parthood verdict
decides what may be *inferred* downstream (Problem A1); a wrong name with the right
verdict is a tidy-up, a wrong verdict is a wrong search result.

In [ ]:
def norm_id(text) -> str:
    # TODO
    raise NotImplementedError

PARTHOOD_WORDS = {"true": True, "yes": True, "false": False, "no": False}

def partwhole_scorer(gold, pred) -> ev.ScoreReport:
    part = norm_id(getattr(pred, "part_category", ""))
    whole = norm_id(getattr(pred, "whole_category", ""))
    relation = norm_id(getattr(pred, "relation", ""))
    parthood = norm_id(getattr(pred, "is_parthood", ""))
    # TODO: the two stages from the table
    raise NotImplementedError

In [ ]:
# Marking scheme for B1 -- do not edit.
with GRADER.check('B1', points=12):
    assert norm_id("Amount of Matter") == "amount-of-matter"
    assert norm_id(" `constituted_of`. ") == "constituted-of"
    marble = next(ex for ex in dev if ex.id == "marble-bust")
    hoard = next(ex for ex in dev if ex.id == "coin-hoard")
    P = lambda p, w, r, h: dspy.Prediction(part_category=p, whole_category=w, relation=r, is_parthood=h)
    cases = [
        (marble, P("amount-of-matter", "physical-object", "constituted-of", "false"), 1.0, []),
        (marble, P("Amount of Matter", "physical_object", " Constituted-Of ", "No"), 1.0, []),
        (marble, P("amount-of-matter", "physical-object", "component-of", "true"), 0.2,
         ["constitution-not-parthood", "check-genuine-parthood"]),
        (marble, P("amount-of-matter", "physical-object", "constituted-of", "true"), 0.6,
         ["check-genuine-parthood"]),
        (marble, P("physical-object", "physical-object", "constituted-of", "false"), 0.9,
         ["identify-categories-first"]),
        (marble, P("amount-of-matter", "physical-object", "part-of", "true"), 0.0, ["use-the-menus"]),
        (marble, P("amount-of-matter", "physical-object", "constituted-of", "maybe"), 0.0,
         ["use-the-menus"]),
        (marble, P("stuff", "physical-object", "constituted-of", "false"), 0.0, ["use-the-menus"]),
        (marble, P("", "", "", ""), 0.0, ["use-the-menus"]),
        (hoard, P("physical-object", "physical-object", "component-of", "true"), 0.5,
         ["identify-categories-first", "member-for-collections"]),
        (hoard, P("physical-object", "collection", "member-of", "false"), 0.6,
         ["check-genuine-parthood"]),
    ]
    for gold, pred, score, violated in cases:
        r = partwhole_scorer(gold, pred)
        assert abs(r.score - score) < 1e-9 and r.violated == violated, (gold.id, pred, r.score, r.violated)
        assert all(v in A.PARTWHOLE_RULEBOOK for v in r.violated)
    wrong = partwhole_scorer(marble, cases[2][1])
    assert ch6.relation_by_id("constituted-of").test in "\n".join(wrong.notes), "name the test question"
    assert any("contradicts" in n for n in partwhole_scorer(marble, cases[3][1]).notes)
    assert not any("contradicts" in n for n in partwhole_scorer(marble, cases[0][1]).notes)

### B2. The classifier as a DSPy program (7 pts)

Write a signature `PartWholeClassification` with inputs `statement`, `categories`,
`relations` and outputs `part_category`, `whole_category`, `relation`, `is_parthood`,
and a factory `PartWholeClassifier(instruction)` returning a `dspy.Module` with a single
`dspy.Predict` whose instruction is `instruction`. The field descriptions are part of the
prompt — write them as you would brief a new cataloguer (manual marks). Configure Claude
and run the program once on `train[0]`.

In [ ]:
BASELINE_INSTRUCTION = A.BASELINE_INSTRUCTION

# TODO: class PartWholeClassification(dspy.Signature): ...
# TODO: def PartWholeClassifier(instruction=BASELINE_INSTRUCTION): ...

lm = llm.configure_dspy()
smoke = None     # TODO: PartWholeClassifier()(**train[0].inputs())

raise NotImplementedError

In [ ]:
# Marking scheme for B2 -- do not edit.
with GRADER.check('B2', points=4):
    assert set(PartWholeClassification.input_fields) == {"statement", "categories", "relations"}
    assert {"part_category", "whole_category", "relation", "is_parthood"} <= set(
        PartWholeClassification.output_fields)
    program = PartWholeClassifier("custom instruction")
    assert len(list(program.named_predictors())) == 1
    assert opt.instruction_of(program) == "custom instruction"
    assert all(isinstance(getattr(smoke, f), str) and getattr(smoke, f).strip()
               for f in ("part_category", "whole_category", "relation", "is_parthood"))

### B3. Baseline on the development split, with its cost (9 pts)

Evaluate the baseline program on `dev` with `ev.evaluate_dataset`, inside
`llm.meter(lm)`. Store the result in `baseline_dev` and the cost in `baseline_dev_cost`,
and show the per-item rows **with the model's four answers next to gold** (re-run the
program on each item — DSPy's cache makes that free).

Then write an **error analysis** of *your* run: for each non-perfect item, which
guideline was violated and why you think Claude answered as it did. Separate three
kinds: format problems, genuine modelling errors, and **disagreements with the museum's
convention** (answers a published taxonomy would defend). They call for different fixes.

In [ ]:
# TODO: baseline_dev = ..., baseline_dev_cost = ...

raise NotImplementedError

In [ ]:
# Marking scheme for B3 -- do not edit.
with GRADER.check('B3', points=3):
    assert baseline_dev["n"] == len(dev) == 8
    assert {r["item"] for r in baseline_dev["rows"]} == {ex.id for ex in dev}
    assert 0.0 <= baseline_dev["mean_score"] <= 1.0
    assert {"calls", "usd"} <= set(baseline_dev_cost)

**Your answer:**

*(Write your answer here.)*

---

## Part C — Optimise honestly — and let the taxonomy compete

The rules for this part: **optimise on `train`, select on `dev`, report on `test`** —
and put a cost and a noise estimate next to every number.

### C1. A budgeted GEPA run with a held-out report (10 pts)

1. Build the GEPA feedback metric from your scorer and `A.PARTWHOLE_RULEBOOK`, and a
   separate reflection LM (`llm.reflection_lm()`).
2. Run `opt.run_gepa` on `train` with `valset=dev` and `max_metric_calls=GEPA_BUDGET`
   inside `llm.meter(lm, reflect)`; store the cost in `gepa_cost`.
3. Compare baseline and tuned on **`test`** with `opt.compare` (store as `c1`) and print
   `c1.report()`.
4. Save the tuned instruction to `config.artifacts_dir() /
   "ch06_partwhole_instruction.txt"`.

In writing: read the instruction diff. Which guidelines did GEPA put into words, which
did it miss, did it state the museum's parthood convention, and did it memorise any
training entry?

In [ ]:
GEPA_BUDGET = 60
# TODO: gepa_metric, reflect, tuned (inside llm.meter -> gepa_cost), c1, save the instruction
instruction_path = config.artifacts_dir() / "ch06_partwhole_instruction.txt"

raise NotImplementedError

In [ ]:
# Marking scheme for C1 -- do not edit.
with GRADER.check('C1', points=5):
    ids = lambda rows: {ex.id for ex in rows}
    assert not (ids(test) & (ids(train) | ids(dev))), "the test split must be untouched"
    assert {r["item"] for r in c1.after["rows"]} == ids(test), "report on the test split"
    assert c1.before["n"] == c1.after["n"] == len(test)
    assert instruction_path.is_file()
    assert instruction_path.read_text(encoding="utf-8") == opt.instruction_of(tuned)
    assert {"calls", "usd"} <= set(gepa_cost) and GEPA_BUDGET <= 100

**Your answer:**

*(Write your answer here.)*

### C2. Four designs, one table: who decides the relation? (9 pts)

Add two contenders to baseline and GEPA:

* **guidelines** — `PartWholeClassifier(A.PARTWHOLE_RULEBOOK.as_guidelines(BASELINE_INSTRUCTION))`;
* **taxonomy** — Claude answers *only* the categories and whether the part is removable
  contents (`separable`), and the chapter's procedure decides the rest. Implement
  `complete_from_categories(part, whole, separable) -> dspy.Prediction` (normalise with
  `norm_id`; if both categories are in the menu, `relation = ch6.classify_partwhole(...)`
  and `is_parthood` from the taxonomy as `"true"`/`"false"`; otherwise `relation=""`,
  `is_parthood=""`), and a program `TaxonomyBackedClassifier()` with a single
  `dspy.Predict` over a signature with inputs `statement`, `categories` and outputs
  `part_category`, `whole_category`, `separable`, whose `forward` returns
  `complete_from_categories(...)`.

Evaluate all four on `test`, each inside its own `llm.meter`, and build `c2` with columns
`program` (`"baseline"`, `"guidelines"`, `"gepa"`, `"taxonomy"`), `test_mean`,
`violations`, `eval_usd`, `optimisation_usd` (GEPA's cost for `"gepa"`, else 0) and
`instruction_chars`.

In writing: which design would you deploy for the 41,000-record migration, and what
would change your mind? Consider score, cost per record, what happens when the
convention changes, and the A2 gaps.

In [ ]:
guidelines_instruction = A.PARTWHOLE_RULEBOOK.as_guidelines(BASELINE_INSTRUCTION)

def complete_from_categories(part, whole, separable) -> dspy.Prediction:
    # TODO
    raise NotImplementedError

# TODO: class ArgumentCategories(dspy.Signature) and def TaxonomyBackedClassifier()
c2 = None    # TODO

In [ ]:
# Marking scheme for C2 -- do not edit.
with GRADER.check('C2', points=3):
    p = complete_from_categories("Amount of Matter", "physical_object", "false")
    assert (p.relation, p.is_parthood) == ("constituted-of", "false")
    p = complete_from_categories("physical-object", "physical-object", "yes")
    assert (p.relation, p.is_parthood) == ("contained-in", "false")
    p = complete_from_categories("physical-object", "collection", "no")
    assert (p.relation, p.is_parthood) == ("member-of", "true")
    assert complete_from_categories("stuff", "region", "no").relation == ""
    assert partwhole_scorer(next(ex for ex in test if ex.id == "silver-salver"),
                            complete_from_categories("amount-of-matter", "physical-object", "no")).score == 1.0
    program = TaxonomyBackedClassifier()
    assert len(list(program.named_predictors())) == 1
    assert isinstance(c2, pd.DataFrame)
    assert set(c2["program"]) == {"baseline", "guidelines", "gepa", "taxonomy"}
    assert {"test_mean", "violations", "eval_usd", "optimisation_usd", "instruction_chars"} <= set(c2.columns)
    assert c2.set_index("program").loc[["baseline", "guidelines", "taxonomy"], "optimisation_usd"].eq(0).all()

**Your answer:**

*(Write your answer here.)*

### C3. Is the difference bigger than the noise? (7 pts)

With caching **disabled** (`fresh = llm.dspy_lm(cache=False)`, used via
`dspy.context(lm=fresh)`), run the GEPA-tuned and the taxonomy-backed programs on `test`
three times each, inside `llm.meter(fresh)` (store as `c3_cost`); record the mean
scores in `runs = {"gepa": [..3..], "taxonomy": [..3..]}`.

Set `verdict_c3` to `"significant"` if `|mean_taxonomy − mean_gepa| > 2 ·
max(sd_gepa, sd_taxonomy)` (sample standard deviations), else `"not significant"`. In
writing: what does this say about your C2 decision, and what would a credible
evaluation of the migration classifier need before it runs on 41,000 records?

In [ ]:
fresh = llm.dspy_lm(cache=False)
runs = {"gepa": [], "taxonomy": []}
# TODO: fill runs (inside llm.meter(fresh) as c3_cost), then compute verdict_c3
verdict_c3 = None

raise NotImplementedError

In [ ]:
# Marking scheme for C3 -- do not edit.
with GRADER.check('C3', points=4):
    import statistics as _st
    assert set(runs) == {"gepa", "taxonomy"} and all(len(v) == 3 for v in runs.values())
    assert all(0.0 <= x <= 1.0 for v in runs.values() for x in v)
    assert {"calls", "usd"} <= set(c3_cost)
    _m = {k: _st.mean(v) for k, v in runs.items()}
    _s = {k: _st.stdev(v) for k, v in runs.items()}
    _expected = ("significant" if abs(_m["taxonomy"] - _m["gepa"]) > 2 * max(_s.values())
                 else "not significant")
    assert verdict_c3 == _expected

**Your answer:**

*(Write your answer here.)*

---

## Part D — A chaining agent, and the curator interview as an MDP

Two deliverables for the search and cataloguing teams. First, an agent that answers
*"is X part of Y?"* over the typed links, with every chain checked by the chapter's
`can_chain`. Second, a plan for aligning *new* catalogue classes to a foundational
category by asking a curator yes/no questions — modelled as a stochastic MDP whose
optimal policy is a decision tree.

In [ ]:
ws_demo = A.CatalogueWorkspace()
tools_demo = {t.name: t for t in A.build_catalogue_tools(ws_demo)}
print(tools_demo["trace_upward"].invoke({"item": "pendulum"}))
print(tools_demo["check_chaining"].invoke({"first": "component-of", "second": "member-of"}))
print("tool calls logged:", ws_demo.log.names())
pd.DataFrame([{"id": q, "question": text, "part": p, "whole": w, "gold": g}
              for q, p, w, text, g in A.CHAIN_QUERIES])

### D1. Build and evaluate the chaining agent (12 pts)

1. Write `CHAINING_PROMPT`, the agent's system prompt. The agent must trace the path
   from the part upwards with its tools, answer *no* if the whole is not on the path,
   decide a single link from the relation's parthood flag, check **every consecutive
   pair** of a longer path with `check_chaining`, and end with a JSON object
   `{"verdict": "yes" | "no", "relations": [...the relations on the path...], "reason": "..."}`.
2. Write `parse_answer(text) -> dict` returning `{"verdict": ..., "relations": [...]}`
   from the agent's final text (the JSON may be fenced or surrounded by prose; use the
   last object that has a `verdict`); the verdict is `"yes"`, `"no"` or `"unparseable"`,
   and `relations` is a list of `norm_id`-ed strings (empty if absent).
3. For every query in `A.CHAIN_QUERIES`, build a fresh `A.CatalogueWorkspace()`, its
   tools, an agent (`agents.build_agent(ws, system_prompt=..., tools=...)`), run it on
   the question followed by the two catalogue ids, and collect `d1` with columns `id`,
   `gold`, `predicted`, `correct`, `relations`, `path_ok` (the reported relations equal
   the true path from part to whole), `tool_calls`, `chain_checks` (number of
   `check_chaining` calls) and `usd_estimate` (`llm.chat_usage(run.messages)`).

In writing: report accuracy with its sample size and the cost per question, and examine
every error, every `path_ok == False` and every multi-link query answered without a
chain check. Then the uncomfortable question: once the links are typed, does this task
need an LLM at all?

In [ ]:
CHAINING_PROMPT = """TODO"""

def parse_answer(text: str) -> dict:
    # TODO
    raise NotImplementedError

d1 = None   # TODO: run the agent on every query

In [ ]:
# Marking scheme for D1 -- do not edit.
with GRADER.check('D1', points=5):
    assert parse_answer('{"verdict": "yes", "relations": ["sub-quantity-of", "sub-quantity-of"], '
                        '"reason": "x"}') == {"verdict": "yes",
                                              "relations": ["sub-quantity-of", "sub-quantity-of"]}
    assert parse_answer('Checked.\n```json\n{"verdict": "No", "relations": ["Component of", '
                        '"member_of"], "reason": "x"}\n```') == {
        "verdict": "no", "relations": ["component-of", "member-of"]}
    assert parse_answer('prose {"note": 1} then {"verdict": "yes"}')["verdict"] == "yes"
    assert parse_answer('{"verdict": "maybe", "relations": []}')["verdict"] == "unparseable"
    assert parse_answer("I think yes.") == {"verdict": "unparseable", "relations": []}
    assert isinstance(d1, pd.DataFrame) and len(d1) == len(A.CHAIN_QUERIES)
    assert {"id", "gold", "predicted", "correct", "relations", "path_ok", "tool_calls",
            "chain_checks", "usd_estimate"} <= set(d1.columns)
    assert set(d1["predicted"]) <= {"yes", "no", "unparseable"}
    assert (d1["correct"] == (d1["predicted"] == d1["gold"])).all()
    _ws = A.CatalogueWorkspace()
    for (qid, part, whole, _, _), (_, row) in zip(A.CHAIN_QUERIES, d1.iterrows()):
        path = []
        for link in _ws.upward(part):
            path.append(link["relation"])
            if link["whole"] == whole:
                break
        assert row["path_ok"] == (list(row["relations"]) == path), qid

**Your answer:**

*(Write your answer here.)*

### D2. Deriving the decision tree — and testing the claim (8 pts)

`A.CategoryDiagnosisMDP` aligns a new class to one of seven DOLCE-style categories. The
state is the set of categories still consistent with the curator's answers; the actions
ask a decision question that splits the set, or commit; the answer is **stochastic**
(probability = prior mass of the categories giving it); each question costs
`question_cost`, and committing earns the probability of being right.

1. Solve the uniform-prior MDP at `question_cost=0.05` with `mdp.value_iteration`; print
   the tree (`M.decision_tree(pi)`). Compute the Q-value of every action at the root in
   `root_q` (`{action: value}`) and set `tied_first` to the sorted list of questions
   whose root Q-value is within `1e-9` of the best. Implement `expected_questions(M,
   policy)` — the exact expected number of questions the policy asks (recurse over the
   tree; do not simulate).
2. A colleague argues: "under a uniform prior one question raises accuracy by at most
   1/7, so a curator should never be asked once a question costs ≥ 1/7". Derive the
   actual threshold `c*` above which committing blind beats the full tree, as a function
   of the number of categories `k` and the expected number of questions `q` of the full
   tree; implement it as `blind_threshold(q, k)`.
3. Verify: sweep `question_cost` over `np.linspace(0, 0.5, 51)` and record in `d2`
   (columns `question_cost`, `value`, `questions_in_tree`, `root_action`).

In writing: why is the colleague wrong? And what does `tied_first` do to the claim that
the MDP "derives DOLCE's decision tree"?

In [ ]:
M = A.CategoryDiagnosisMDP(question_cost=0.05)
# TODO (1): V, pi, root_q, tied_first, expected_questions

def blind_threshold(q: float, k: int) -> float:
    # TODO (2)
    raise NotImplementedError

d2 = None     # TODO (3)

In [ ]:
# Marking scheme for D2 -- do not edit.
with GRADER.check('D2', points=4):
    assert abs(V[M.initial_state()] - 0.85) < 1e-9
    assert set(root_q) == set(M.actions(M.initial_state()))
    assert tied_first == ["dependent", "happens", "spatial"]
    assert abs(expected_questions(M, pi) - 3.0) < 1e-9
    assert abs(blind_threshold(3, 7) - 2 / 7) < 1e-12 and abs(blind_threshold(2, 4) - 0.375) < 1e-12
    assert isinstance(d2, pd.DataFrame) and len(d2) == 51
    c_star = blind_threshold(expected_questions(M, pi), 7)
    below, above = d2[d2.question_cost < c_star - 0.005], d2[d2.question_cost > c_star + 0.005]
    assert (below.questions_in_tree == 6).all() and (below.root_action != "commit").all()
    assert (above.questions_in_tree == 0).all() and (above.root_action == "commit").all()

**Your answer:**

*(Write your answer here.)*

### D3. The museum's interview plan: prior, curator minutes, and a missing question (6 pts)

The museum's new classes are not uniform over categories (`A.CATALOGUE_CLASS_PRIOR`),
and its curators do not find all questions equally easy (`A.CURATOR_MINUTES`, median
minutes per reliable answer). Value one curator minute at `RATE = 0.01` (a hundredth of a
correct alignment).

1. Implement `evaluate_tree(M, policy, minutes) -> (accuracy, expected_minutes)`, exact
   by recursion, where `minutes` gives the curator minutes of each question.
2. Build four settings, each solved with value iteration: `"textbook"` (uniform prior,
   flat cost 0.05), `"museum-flat"` (museum prior, flat cost 0.05), `"museum-timed"`
   (museum prior, cost `RATE · minutes` per question) and `"museum-timed-no-happens"`
   (as museum-timed, but the `happens` question is not available). Evaluate **every
   setting's policy under the museum-timed MDP where possible** — i.e. the real prior —
   except `"museum-timed-no-happens"`, which is evaluated in its own MDP. Build `d3`
   (index: setting) with columns `root_question`, `accuracy`, `expected_minutes`, `value`
   (V*(s0) of the setting's own MDP).
3. Set `inseparable` to the set of categories that no remaining question can tell apart
   once `happens` is removed.

In writing: which interview plan would you give the cataloguers, and what does this
say about *why* foundational ontologies start with the endurant/perdurant distinction?
What does the MDP still leave out?

In [ ]:
RATE = 0.01

def evaluate_tree(M, policy, minutes, state=None):
    # TODO (1)
    raise NotImplementedError

d3 = None           # TODO (2)
inseparable = None  # TODO (3)

In [ ]:
# Marking scheme for D3 -- do not edit.
with GRADER.check('D3', points=3):
    assert isinstance(d3, pd.DataFrame)
    assert set(d3.index) == {"textbook", "museum-flat", "museum-timed", "museum-timed-no-happens"}
    assert {"root_question", "accuracy", "expected_minutes", "value"} <= set(d3.columns)
    assert d3.loc["museum-flat", "root_question"] == "dependent"
    assert d3.loc["museum-timed", "root_question"] == "happens"
    assert abs(d3.loc["museum-timed", "value"]
               - (d3.loc["museum-timed", "accuracy"] - RATE * d3.loc["museum-timed", "expected_minutes"])) < 1e-3
    assert d3.loc["museum-timed", "expected_minutes"] < d3.loc["museum-flat", "expected_minutes"]
    assert d3.loc["museum-timed-no-happens", "value"] < d3.loc["museum-timed", "value"]
    assert d3.loc["museum-timed-no-happens", "accuracy"] < 1.0
    assert inseparable == {"physical-object", "process"}
    _Mt = A.CategoryDiagnosisMDP({q: RATE * m for q, m in A.CURATOR_MINUTES.items()},
                                 prior=A.CATALOGUE_CLASS_PRIOR)
    _acc, _min = evaluate_tree(_Mt, {}, A.CURATOR_MINUTES)
    assert abs(_acc - 0.52) < 1e-9 and _min == 0.0, "an empty policy commits at once"

**Your answer:**

*(Write your answer here.)*

---

## Grading rubric

| Part | Problem | Auto | Manual | Pts |
|---|---|---|---|---|
| A | A1 Every spurious answer the search gives | 6 | 4 | 10 |
| A | A2 Where the categories stop deciding | 4 | 4 | 8 |
| B | B1 A diagnostic scorer that separates naming from parthood | 12 | 2 | 14 |
| B | B2 The classifier as a DSPy program | 4 | 3 | 7 |
| B | B3 Baseline on the development split, with its cost | 3 | 6 | 9 |
| C | C1 A budgeted GEPA run with a held-out report | 5 | 5 | 10 |
| C | C2 Four designs, one table: who decides the relation? | 3 | 6 | 9 |
| C | C3 Is the difference bigger than the noise? | 4 | 3 | 7 |
| D | D1 Build and evaluate the chaining agent | 5 | 7 | 12 |
| D | D2 Deriving the decision tree — and testing the claim | 4 | 4 | 8 |
| D | D3 The museum's interview plan: prior, curator minutes, and a missing question | 3 | 3 | 6 |
| **Total** | | | | **100** |

*Manual* points are for the written answers and for the quality of the engineering evidence (splits, sample sizes, cost, honest negative results).

---

## Self-test

Re-run this cell after finishing. It lists every auto-graded problem and whether its check passed.

In [ ]:
AUTO_GRADED = ['A1', 'A2', 'B1', 'B2', 'B3', 'C1', 'C2', 'C3', 'D1', 'D2', 'D3']
_ = GRADER.summary(AUTO_GRADED)